# Gold template: Type 2 dimension (Snowflake)

Builds a dimension that keeps history: when a tracked attribute changes, the current version is closed and a
new version is added. Facts point at the version that was current on the fact's date.

**Standards this notebook follows**

1. **Two keys.** `<NAME>_SK` is the version key facts join on; its default is the dimension's own
   **sequence** (`<TABLE>_SEQ.NEXTVAL`). `<NAME>_DK`, the durable key, is the Silver key: the same for every version.
2. **Validity.** `VALID_FROM` (inclusive), `VALID_TO` (exclusive) and `IS_CURRENT`. The first version starts at
   `1900-01-01`; the current one ends at `9999-12-31`. A fact on day `d` takes `VALID_FROM <= d < VALID_TO`.
3. **Tracked and overwritten attributes.** A change in `type2_columns` makes a new version; a change in
   `type1_columns` overwrites every version.
4. **One atomic MERGE** closes changed versions and adds new ones; a re-run with no changes does nothing.
5. **Unknown member** `-1`, valid for all dates, with `Unknown` in its text columns.
6. **Audit columns:** `_TYPE2_HASH`, `_TYPE1_HASH`, `_RUN_ID`, `_LOADED_AT`. **Layers.** one database per layer: `BRONZE`, `SILVER`, `GOLD`, with keys in `KEYS`. Tables are named `<DATABASE>.<SCHEMA>.<TABLE>`.

## Who supports this notebook

This is an **enterprise template** (`gold-dim-type2` 1.0.0, Snowflake). The line between enterprise and domain is drawn by the cells:

| Cells | Marked | Who owns them |
| --- | --- | --- |
| **Parameters** | `# kratosforge:parameters` | The **domain**: table names, keys, columns, measures. Values only, no code. |
| Every other code cell | `# kratosforge:standard` | The **enterprise**: it maintains, fixes and upgrades them. Don't edit them. |
| Markdown cells | none | Anyone: add notes freely. |

While the standard cells are unchanged and no code cell is added, the notebook **conforms** and the enterprise can
take it over: check it with `fabric-desktop templates check <notebook>`, then hand it over with
`fabric-desktop templates handover <notebook> --domain <domain> --owner <you>`.
Editing a standard cell, or adding a code cell, makes it **domain-owned** ad hoc work: it still runs, but the
domain supports it, and the check lists each change. Need logic the template doesn't have? Put it in Silver
(a derived column) or ask the enterprise to add it to the template, so every domain gets it.

In [ ]:
# kratosforge:parameters template=gold-dim-type2 platform=snowflake version=1.0.0 | The domain fills in this cell: values only, no code.
# Parameters
source_table = "SILVER.CRM.CUSTOMERS"
target_table = "GOLD.SALES.DIM_CUSTOMER"
source_key_column = "customer_sk"    # the Silver key from the key table
sk_column = "customer_sk"            # the version key, issued here; facts join on it
durable_key_column = "customer_dk"   # the Silver key, the same for every version
business_key = {"customer_id": "number(10,0)"}
type2_columns = {"segment": "varchar", "city": "varchar"}   # a change makes a new version
type1_columns = {"name": "varchar", "email": "varchar"}     # a change overwrites every version
load_date = ""                       # the date changes take effect (yyyy-mm-dd); today (UTC) when empty
run_id = ""          # the task or orchestrator passes its run id; one is made when empty
full_reload = False  # True re-reads the whole source (after a logic change or a backfill)

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=snowflake version=1.0.0 cell=setup | Enterprise-managed: change the Parameters cell, not this one.
import re
import uuid
from datetime import datetime, timezone

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException

session = get_active_session()
UNKNOWN_SK = -1  # the Unknown member: rows whose key can't be found point here, never at NULL
_NAME = re.compile(r"[A-Za-z_][A-Za-z0-9_$]{0,254}")
_TYPE = re.compile(r"(?i)(varchar|string|text|boolean|number\(\d{1,2}(,\s*\d{1,2})?\)|int|integer|bigint|float|double|date|timestamp_ntz|timestamp_tz)")
_RUN_ID = re.compile(r"[A-Za-z0-9_.:-]{1,64}")


def q(name: str) -> str:
    """A table name, checked: parameters arrive from tasks, so nothing else can reach the SQL."""
    parts = name.split(".")
    if not 1 <= len(parts) <= 3 or not all(_NAME.fullmatch(x) for x in parts):
        raise ValueError(f"Not a table name: {name!r}")
    return ".".join(parts).upper()


def c(name: str) -> str:
    if not _NAME.fullmatch(name):
        raise ValueError(f"Not a column name: {name!r}")
    return name.upper()


def checked_type(dtype: str) -> str:
    if not _TYPE.fullmatch(dtype.strip()):
        raise ValueError(f"Not a supported type: {dtype!r}")
    return dtype.strip().upper()


def run(sql: str) -> list:
    """Runs one statement. DML and DDL return one summary row; collect() is how Snowpark runs them."""
    return session.sql(sql).collect()


def table_exists(name: str) -> bool:
    try:
        session.table(q(name)).schema  # resolved now, so a missing table raises here
        return True
    except SnowparkSQLException:
        return False


def row_hash(columns: list, alias: str = "") -> str:
    """SHA-256 of the columns, NULL-safe: a change in any of them changes the hash."""
    a = f"{alias}." if alias else ""
    parts = [f"COALESCE(TO_VARCHAR({a}{c(x)}), CHR(0))" for x in columns] or ["''"]
    return f"SHA2(ARRAY_TO_STRING(ARRAY_CONSTRUCT({', '.join(parts)}), CHR(31)), 256)"


def is_text(dtype: str) -> bool:
    return dtype.upper().split("(")[0] in ("VARCHAR", "STRING", "TEXT")


run_id = run_id or str(uuid.uuid4())
if not _RUN_ID.fullmatch(run_id):
    raise ValueError("run_id may only hold letters, digits and _ . : -")
loaded_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
LOADED_AT = f"TO_TIMESTAMP_NTZ('{loaded_at}')"
print(f"Run {run_id} at {loaded_at} UTC")

## 1. Create the table, its sequence and its Unknown member

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=snowflake version=1.0.0 cell=step-1 | Enterprise-managed: change the Parameters cell, not this one.
attributes = {**business_key, **type2_columns, **type1_columns}
if len(attributes) != len(business_key) + len(type2_columns) + len(type1_columns):
    raise ValueError("A column can be in only one of business_key, type2_columns and type1_columns")
attributes = {c(n): checked_type(t) for n, t in attributes.items()}
type2 = [c(n) for n in [*business_key, *type2_columns]]
type1 = [c(n) for n in type1_columns]
sk, dk, src_key = c(sk_column), c(durable_key_column), c(source_key_column)
load_date = load_date or loaded_at[:10]
if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", load_date):
    raise ValueError("load_date must look like 2026-11-05")
sequence = f"{q(target_table)}_SEQ"
every = [sk, dk, *attributes, "VALID_FROM", "VALID_TO", "IS_CURRENT", "_TYPE2_HASH", "_TYPE1_HASH", "_RUN_ID", "_LOADED_AT"]
def unknown_member_values(types: dict) -> list:
    """The Unknown member's value for each column: 'Unknown' for text, NULL for the rest."""
    return ["'Unknown'" if is_text(t) else f"CAST(NULL AS {t})" for t in types.values()]

run(f"CREATE SEQUENCE IF NOT EXISTS {sequence} START = 1 INCREMENT = 1 ORDER")
run(f"""CREATE TABLE IF NOT EXISTS {q(target_table)} (
    {", ".join([f"{sk} NUMBER(10,0) NOT NULL DEFAULT {sequence}.NEXTVAL", f"{dk} NUMBER(10,0) NOT NULL",
                *[f"{n} {t}" for n, t in attributes.items()],
                "VALID_FROM DATE NOT NULL", "VALID_TO DATE NOT NULL", "IS_CURRENT BOOLEAN NOT NULL",
                "_TYPE2_HASH VARCHAR(64)", "_TYPE1_HASH VARCHAR(64)", "_RUN_ID VARCHAR", "_LOADED_AT TIMESTAMP_NTZ"])},
    CONSTRAINT PK_{sk} PRIMARY KEY ({sk}))""")

unknown = [str(UNKNOWN_SK), str(UNKNOWN_SK), *unknown_member_values(attributes),
           "TO_DATE('1900-01-01')", "TO_DATE('9999-12-31')", "TRUE",
           "'unknown-member'", "'unknown-member'", f"'{run_id}'", LOADED_AT]
run(f"""MERGE INTO {q(target_table)} AS t
USING (SELECT {", ".join(f"{x} AS {n}" for x, n in zip(unknown, every))}) AS s
ON t.{sk} = s.{sk}
WHEN NOT MATCHED THEN INSERT ({", ".join(every)}) VALUES ({", ".join(f"s.{n}" for n in every)})""")

## 2. Close changed versions, add new ones, apply corrections

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=snowflake version=1.0.0 cell=step-2 | Enterprise-managed: change the Parameters cell, not this one.
run(f"""CREATE OR REPLACE TEMPORARY TABLE DIM_STAGED AS
SELECT CAST({src_key} AS NUMBER(10,0)) AS {dk}, {", ".join(f"CAST({n} AS {t}) AS {n}" for n, t in attributes.items())},
       {row_hash(type2)} AS _TYPE2_HASH, {row_hash(type1)} AS _TYPE1_HASH,
       '{run_id}' AS _RUN_ID, {LOADED_AT} AS _LOADED_AT
FROM {q(source_table)} WHERE {src_key} <> {UNKNOWN_SK}""")

# Each staged row against its current version. A new durable key inserts a first version from 1900-01-01.
# A changed tracked attribute appears twice: once with its durable key, to close the current version,
# and once with no merge key, so it inserts as the new current version from load_date.
staged_cols = [dk, *attributes, "_TYPE2_HASH", "_TYPE1_HASH", "_RUN_ID", "_LOADED_AT"]
s_cols = ", ".join(f"s.{n}" for n in staged_cols)
run(f"""CREATE OR REPLACE TEMPORARY TABLE DIM_CHANGES AS
WITH cur AS (SELECT {dk}, _TYPE2_HASH FROM {q(target_table)} WHERE IS_CURRENT AND {sk} <> {UNKNOWN_SK})
SELECT s.{dk} AS _MERGE_KEY, {s_cols}, TO_DATE('1900-01-01') AS _VALID_FROM
FROM DIM_STAGED s LEFT JOIN cur ON cur.{dk} = s.{dk}
WHERE cur.{dk} IS NULL OR cur._TYPE2_HASH <> s._TYPE2_HASH
UNION ALL
SELECT NULL AS _MERGE_KEY, {s_cols}, TO_DATE('{load_date}') AS _VALID_FROM
FROM DIM_STAGED s JOIN cur ON cur.{dk} = s.{dk} AND cur._TYPE2_HASH <> s._TYPE2_HASH""")

inserted = [dk, *attributes, "VALID_FROM", "VALID_TO", "IS_CURRENT", "_TYPE2_HASH", "_TYPE1_HASH", "_RUN_ID", "_LOADED_AT"]
values = [f"s.{dk}", *[f"s.{n}" for n in attributes], "s._VALID_FROM", "TO_DATE('9999-12-31')", "TRUE",
          "s._TYPE2_HASH", "s._TYPE1_HASH", "s._RUN_ID", "s._LOADED_AT"]
result = run(f"""MERGE INTO {q(target_table)} AS t
USING DIM_CHANGES AS s
ON t.{dk} = s._MERGE_KEY AND t.IS_CURRENT AND t.{sk} <> {UNKNOWN_SK}
WHEN MATCHED THEN
    UPDATE SET t.VALID_TO = TO_DATE('{load_date}'), t.IS_CURRENT = FALSE, t._RUN_ID = s._RUN_ID, t._LOADED_AT = s._LOADED_AT
WHEN NOT MATCHED THEN
    INSERT ({", ".join(inserted)}) VALUES ({", ".join(values)})""")
print(f"Type 2 MERGE: {result[0].as_dict() if result else 'done'}")

# Corrections to overwritten attributes go to every version of the row.
if type1:
    run(f"""MERGE INTO {q(target_table)} AS t
    USING DIM_STAGED AS s
    ON t.{dk} = s.{dk} AND t._TYPE1_HASH <> s._TYPE1_HASH
    WHEN MATCHED THEN UPDATE SET {", ".join(f"t.{n} = s.{n}" for n in [*type1, "_TYPE1_HASH", "_RUN_ID", "_LOADED_AT"])}""")

## 3. Check the result

In [ ]:
# kratosforge:standard template=gold-dim-type2 platform=snowflake version=1.0.0 cell=step-3 | Enterprise-managed: change the Parameters cell, not this one.
summary = run(f"""SELECT COUNT(*) AS ROW_COUNT, COUNT(DISTINCT {sk}) AS KEY_COUNT,
       COUNT_IF({sk} = {UNKNOWN_SK}) AS UNKNOWN_MEMBERS, COUNT_IF(VALID_FROM > VALID_TO) AS BAD_RANGES
FROM {q(target_table)}""")[0]
# Exactly one current version per durable key, and versions of one key never overlap.
broken = run(f"""SELECT {dk} FROM (
    SELECT {dk}, IS_CURRENT, VALID_TO,
           LEAD(VALID_FROM) OVER (PARTITION BY {dk} ORDER BY VALID_FROM, {sk}) AS NEXT_FROM
    FROM {q(target_table)} WHERE {sk} <> {UNKNOWN_SK})
GROUP BY {dk}
HAVING COUNT_IF(IS_CURRENT) <> 1 OR COUNT_IF(NEXT_FROM < VALID_TO) > 0
LIMIT 5""")
if summary["ROW_COUNT"] != summary["KEY_COUNT"] or summary["UNKNOWN_MEMBERS"] != 1 or summary["BAD_RANGES"] or broken:
    raise RuntimeError(f"{target_table} breaks a Type 2 rule: {summary.as_dict()}, broken durable keys {broken}")
print(f"{target_table}: {summary['ROW_COUNT']} versions including the Unknown member")